[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc2_donnees/corrections/seance3_correction.ipynb)

# Séance 2.3 — Agréger et croiser plusieurs tables

**Correction** · durée : 2h (≈50 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- filtrer sur plusieurs conditions sans se noyer dans les parenthèses
- classer et extraire un top 5 en une commande
- répondre à « combien par ... ? » avec `groupby`
- calculer plusieurs indicateurs d'un coup avec `agg`
- rassembler trois fichiers en une seule table avec `merge`
- croiser deux dimensions avec un tableau croisé

## Correction

Solutions commentées. Comparez avec ce que vous aviez écrit : plusieurs formulations peuvent être correctes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc2_donnees/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

Chargement des données utilisées dans toute la feuille :

In [ ]:
ventes = pd.read_csv(BASE + "ventes.csv")
clients = pd.read_csv(BASE + "clients.csv")
produits = pd.read_csv(BASE + "produits.csv")

ventes["ca"] = ventes["qte"] * ventes["prix"]
print(ventes.shape, clients.shape, produits.shape)

### Exercice 1 — Filtrer sur deux conditions

> **Votre mission :**
> - Garder les ventes dont le prix dépasse 10 € **et** la quantité dépasse 20.
> - Mettre le nombre de lignes dans `nb_premium`.

In [ ]:
# Dans query(), on combine avec "and" (et non le & des crochets)
premium = ventes.query("prix > 10 and qte > 20")
nb_premium = len(premium)

print(nb_premium)

In [ ]:
verifier("1 - ventes cheres et volumineuses", nb_premium == 38,
         "dans query() on ecrit and, pas &")

### Exercice 2 — Filtrer sur une liste de pays

> **Votre mission :**
> - Compter les clients situés en France, en Allemagne ou en Belgique → `nb_ue`.
> - ⚠️ Guillemets doubles à l'extérieur, simples à l'intérieur.

In [ ]:
# "in" teste l'appartenance a une liste : bien plus court
# que pays == 'France' or pays == 'Allemagne' or ...
nb_ue = len(clients.query("pays in ['France', 'Allemagne', 'Belgique']"))

print(nb_ue)

In [ ]:
verifier("2 - clients dans trois pays", nb_ue == 121,
         "le mot-cle est in, et les noms de pays vont entre guillemets simples")

### Exercice 3 — Le meilleur client

> **Votre mission :**
> - Calculer le chiffre d'affaires par client.
> - Mettre l'identifiant du meilleur dans `meilleur_client` et son CA dans `ca_meilleur` (arrondi à 2 décimales).

In [ ]:
ca_client = ventes.groupby("client_id")["ca"].sum()

# idxmax() donne l'etiquette du maximum, max() donne sa valeur
meilleur_client = ca_client.idxmax()
ca_meilleur = round(ca_client.max(), 2)

print(meilleur_client, ":", ca_meilleur, "euros")

In [ ]:
verifier("3a - meilleur client", meilleur_client == 14911,
         "groupby sur client_id puis sum() sur la colonne ca")
verifier("3b - son chiffre d'affaires", ca_meilleur == 143825.06,
         "idxmax() renvoie l'identifiant, max() renvoie le montant")

### Exercice 4 — Lignes contre commandes

> **Votre mission :**
> - Pour chaque client, calculer le nombre de **lignes** (`nb_lignes`) et le nombre de **commandes distinctes** (`nb_cmd`).
> - Mettre le résultat dans `resume`.
> - Rappel : une commande de 30 articles = 30 lignes, mais 1 commande.

In [ ]:
resume = ventes.groupby("client_id").agg(
    nb_lignes=("cmd_id", "count"),    # compte les lignes
    nb_cmd=("cmd_id", "nunique"),     # compte les valeurs DISTINCTES
)

print(resume["nb_lignes"].sum(), "lignes |", resume["nb_cmd"].sum(), "commandes")

In [ ]:
verifier("4a - nombre de lignes", resume["nb_lignes"].sum() == 45123,
         "count compte les lignes du groupe")
verifier("4b - nombre de commandes", resume["nb_cmd"].sum() == 1955,
         "nunique compte les valeurs distinctes, count compte les lignes")

### Exercice 5 — La jointure

> **Votre mission :**
> - Joindre `ventes` et `clients` sur `client_id` → `vc`.
> - **Vérifier** que le nombre de lignes n'a pas changé → `nb_apres`.

In [ ]:
vc = ventes.merge(clients, on="client_id")
nb_apres = len(vc)

# Le reflexe a ne jamais sauter : un merge peut dupliquer
# ou faire disparaitre des lignes sans rien dire
print(len(ventes), "->", nb_apres)

In [ ]:
verifier("5 - jointure sans perte", nb_apres == 45123,
         "un nombre different signale une cle de jointure non unique")

### Exercice 6 — Le chiffre d'affaires par pays

> **Votre mission :**
> - À partir de `vc`, calculer le CA par pays, trié du plus grand au plus petit → `ca_pays`.
> - Mettre le CA de la France dans `ca_france` (arrondi à 2 décimales).

In [ ]:
ca_pays = vc.groupby("pays")["ca"].sum().sort_values(ascending=False)

# On accede a une valeur par son etiquette, comme dans un dictionnaire
ca_france = round(ca_pays["France"], 2)

print(ca_pays.head(3).round(2))
print("France :", ca_france)

In [ ]:
verifier("6 - CA de la France", ca_france == 133984.8,
         "groupby('pays') puis sum() sur ca, et ca_pays['France']")

### Exercice 7 — Le panier moyen par pays

> **Votre mission :**
> - Pour chaque pays : le CA total (`ca`) et le nombre de **commandes distinctes** (`nb_cmd`).
> - Ajouter une colonne `panier` = CA ÷ nombre de commandes, arrondie à 2 décimales.
> - Mettre le panier moyen irlandais dans `panier_irl`.

In [ ]:
parpays = vc.groupby("pays").agg(
    ca=("ca", "sum"),
    nb_cmd=("cmd_id", "nunique"),   # des COMMANDES, pas des lignes
)
parpays["panier"] = (parpays["ca"] / parpays["nb_cmd"]).round(2)

# .loc[ligne, colonne] pour aller chercher une case precise
panier_irl = parpays.loc["Irlande", "panier"]
print(panier_irl)

In [ ]:
verifier("7 - panier moyen irlandais", panier_irl == 1020.33,
         "avec count au lieu de nunique le panier serait ridiculement bas")

### Exercice 8 — Ajouter les produits

> **Votre mission :**
> - Joindre `vc` et `produits` sur `prod_id` → `complet`.
> - Trouver la catégorie qui génère le plus de CA → `cat_top`.

In [ ]:
complet = vc.merge(produits, on="prod_id")

# idxmax() renvoie le nom de la categorie, pas son montant
cat_top = complet.groupby("categorie")["ca"].sum().idxmax()

print(len(complet), "lignes | categorie leader :", cat_top)

In [ ]:
verifier("8a - jointure produits", len(complet) == 45123,
         "la cle commune entre vc et produits est prod_id")
verifier("8b - categorie leader", cat_top == "cuisine",
         "groupby('categorie'), sum() sur ca, puis idxmax()")

### Exercice 9 — Le tableau croisé

> **Votre mission :**
> - Croiser `pays` (en lignes) et `segment` (en colonnes), avec la somme du `ca` → `tableau`.
> - Mettre le CA des clients « premium » français dans `fr_premium` (arrondi à 0 décimale).

In [ ]:
# index = ce qui va en lignes, columns = ce qui va en colonnes
tableau = vc.pivot_table(values="ca", index="pays", columns="segment", aggfunc="sum")

fr_premium = round(tableau.loc["France", "premium"], 0)
print(fr_premium)

In [ ]:
verifier("9 - premium francais", fr_premium == 114433.0,
         "index=pays (lignes), columns=segment (colonnes), aggfunc='sum'")

### Exercice 10 — Question de synthèse

> **Votre mission :**
> - Quelle **part du chiffre d'affaires total** l'Irlande représente-t-elle, en % arrondi à 1 décimale ? → `part_irl`
> - Combien de clients irlandais y a-t-il ? → `nb_cli_irl`
> - Regardez les deux chiffres ensemble. Que diriez-vous à un dirigeant ?

In [ ]:
# ca_pays.sum() = le CA total, tous pays confondus
part_irl = round(100 * ca_pays["Irlande"] / ca_pays.sum(), 1)
nb_cli_irl = vc.query("pays == 'Irlande'")["client_id"].nunique()

print(part_irl, "% du CA pour", nb_cli_irl, "clients")

# A retenir pour la seance 2.4 : le deuxieme marche du groupe repose
# entierement sur DEUX comptes. Si l'un des deux part, 11 % du chiffre
# d'affaires disparait. Ce n'est pas un marche, c'est un risque.

In [ ]:
verifier("10a - part de l'Irlande", part_irl == 22.7,
         "divisez le CA irlandais par ca_pays.sum()")
verifier("10b - clients irlandais", nb_cli_irl == 2,
         "nunique() sur client_id apres avoir filtre sur l'Irlande")